In [1]:
import json
import time
from collections import Counter
from functools import lru_cache
from itertools import islice

import pandas as pd

from kyb.config import Paths
from kyb.data.readers import iter_psc
from kyb.data.subset import CORPORATE, INDIVIDUAL, LEGAL, STATEMENT
from kyb.normalize import looks_uk_registered, normalize_company_number, parse_natures

ORNEK_BOYUTU = None   # None yaparsan bütün dosya okunur

psc_dosyalari = sorted((Paths().raw / "ch_psc").glob("*.zip"))
print("Okunacak dosyalar:", [p.name for p in psc_dosyalari])

Okunacak dosyalar: ['persons-with-significant-control-snapshot-2026-11-21.zip']


Ölçüm 2: Aktif ve sona ermiş kayıtlar

In [2]:
tur_sayaci = Counter()

def olcum1(veri, tur):
    tur_sayaci[tur] += 1

In [3]:
sona_eren_sayaci = Counter()
sona_erme_yili = Counter()

def olcum2(veri, tur):
    bitis = veri.get("ceased_on")
    if bitis:
        sona_eren_sayaci[tur] += 1
        sona_erme_yili[str(bitis)[:4]] += 1

In [4]:
ALANLAR = [
    "name", "nationality", "country_of_residence", "date_of_birth",
    "identification", "natures_of_control", "notified_on",
]
dolu_alan_sayaci = Counter()   # (tür, alan) -> dolu kayıt sayısı
anahtar_sayaci = Counter()     # (tür, anahtar) -> görülme sayısı

def olcum3(veri, tur):
    for alan in ALANLAR:
        if veri.get(alan):
            dolu_alan_sayaci[(tur, alan)] += 1
    for anahtar in veri:
        anahtar_sayaci[(tur, anahtar)] += 1

In [5]:
KONTROL_TURLERI = INDIVIDUAL | CORPORATE | LEGAL

ifade_sayaci = Counter()        # her ifadenin görülme sayısı
bilinmeyen_ifade = Counter()    # ayrıştırıcının tanımadığı ifadeler
bilinmeyenli_kayit = Counter()  # tür -> en az 1 bilinmeyen ifadesi olan kayıt
ifadesi_olmayan = Counter()     # tür -> natures_of_control alanı boş kayıt
pay_bilgisi_yok = Counter()     # tür -> ne pay ne oy aralığı olan kayıt

@lru_cache(maxsize=None)
def ayristir(ifadeler: tuple):
    return parse_natures(list(ifadeler))

def olcum4(veri, tur):
    if tur not in KONTROL_TURLERI:
        return
    ifadeler = veri.get("natures_of_control")
    if not ifadeler:
        ifadesi_olmayan[tur] += 1
        return
    for ifade in ifadeler:
        ifade_sayaci[ifade] += 1
    ozet = ayristir(tuple(ifadeler))
    if ozet.unknown:
        bilinmeyenli_kayit[tur] += 1
        for ifade in ozet.unknown:
            bilinmeyen_ifade[ifade] += 1
    if ozet.share_min is None and ozet.vote_min is None:
        pay_bilgisi_yok[tur] += 1

In [6]:
huni = Counter()                # adım -> kayıt sayısı
basarisiz_numara = Counter()    # İngiliz sicili ama normalize edilemeyen ham numaralar
yabanci_yer = Counter()         # İngiliz sayılmayan kayıtların sicil yeri / ülkesi

def olcum5(veri, tur):
    if tur not in CORPORATE or veri.get("ceased_on"):
        return
    huni["1_kurumsal_aktif"] += 1

    kimlik = veri.get("identification") or {}
    if not kimlik:
        return
    huni["2_identification_dolu"] += 1

    if not looks_uk_registered(kimlik):
        yer = kimlik.get("country_registered") or kimlik.get("place_registered") or "<boş>"
        yabanci_yer[str(yer).strip().upper()] += 1
        return
    huni["3_ingiliz_sicili"] += 1

    if normalize_company_number(kimlik.get("registration_number")):
        huni["4_numara_normalize_edildi"] += 1
    else:
        basarisiz_numara[str(kimlik.get("registration_number"))] += 1

In [7]:
aciklama_sayaci = Counter()   # (açıklama, aktif/sona_ermis) -> adet
uyruk_sayaci = Counter()      # uyruk metni -> adet (gerçek kişiler)

def olcum6(veri, tur):
    if tur == STATEMENT:
        durum = "sona_ermis" if veri.get("ceased_on") else "aktif"
        aciklama_sayaci[(veri.get("statement", "<yok>"), durum)] += 1
    elif tur in INDIVIDUAL:
        uyruk_sayaci[str(veri.get("nationality") or "<boş>").strip()] += 1

In [8]:
OLCUMLER = [olcum1, olcum2, olcum3, olcum4, olcum5, olcum6]
SAYACLAR = [
    tur_sayaci, sona_eren_sayaci, sona_erme_yili, dolu_alan_sayaci, anahtar_sayaci,
    ifade_sayaci, bilinmeyen_ifade, bilinmeyenli_kayit, ifadesi_olmayan, pay_bilgisi_yok,
    huni, basarisiz_numara, yabanci_yer, aciklama_sayaci, uyruk_sayaci,
]
for sayac in SAYACLAR:
    sayac.clear()
ayristir.cache_clear()

akis = iter_psc(psc_dosyalari)
if ORNEK_BOYUTU:
    akis = islice(akis, ORNEK_BOYUTU)

toplam = 0
baslangic = time.time()
for _sirket_no, veri in akis:
    tur = veri.get("kind", "<kind yok>")
    for olcum in OLCUMLER:
        olcum(veri, tur)
    toplam += 1
    if toplam % 1_000_000 == 0:
        print(f"{toplam:,} kayıt · {time.time() - baslangic:.0f} sn")

print(f"Bitti: {toplam:,} kayıt · {time.time() - baslangic:.1f} sn")
print("Farklı ifade kombinasyonu:", ayristir.cache_info().currsize)

1,000,000 kayıt · 18 sn
2,000,000 kayıt · 35 sn
3,000,000 kayıt · 52 sn
4,000,000 kayıt · 70 sn
5,000,000 kayıt · 87 sn
6,000,000 kayıt · 104 sn
7,000,000 kayıt · 121 sn
8,000,000 kayıt · 139 sn
9,000,000 kayıt · 156 sn
10,000,000 kayıt · 173 sn
11,000,000 kayıt · 190 sn
12,000,000 kayıt · 207 sn
13,000,000 kayıt · 225 sn
14,000,000 kayıt · 242 sn
15,000,000 kayıt · 260 sn
16,000,000 kayıt · 270 sn
Bitti: 16,178,384 kayıt · 271.4 sn
Farklı ifade kombinasyonu: 9454


In [9]:
tablo = pd.DataFrame(tur_sayaci.most_common(), columns=["kind", "adet"])
tablo["yuzde"] = (tablo["adet"] / toplam * 100).round(2)
tablo

,kind,adet,yuzde
0,individual-person-with-significant-control,14081675,87.04
1,corporate-entity-person-with-significant-control,1091153,6.74
2,persons-with-significant-control-statement,936034,5.79
3,individual-beneficial-owner,34980,0.22
4,legal-person-person-with-significant-control,17395,0.11
5,corporate-entity-beneficial-owner,15891,0.10
6,legal-person-beneficial-owner,592,0.00
7,super-secure-person-with-significant-control,386,0.00
8,super-secure-beneficial-owner,169,0.00
9,exemptions,109,0.00


In [10]:
tablo["sona_ermis"] = tablo["kind"].map(sona_eren_sayaci).fillna(0).astype(int)
tablo["sona_erme_orani"] = (tablo["sona_ermis"] / tablo["adet"] * 100).round(2)

genel_oran = sum(sona_eren_sayaci.values()) / toplam * 100
print(f"Sona ermiş kayıtların genel oranı: %{genel_oran:.2f}")

yillar = pd.DataFrame(sorted(sona_erme_yili.items()), columns=["yil", "adet"])
display(tablo)
display(yillar)

Sona ermiş kayıtların genel oranı: %16.85


,kind,adet,yuzde,sona_ermis,sona_erme_orani
0,individual-person-with-significant-control,14081675,87.04,2094331,14.87
1,corporate-entity-person-with-significant-control,1091153,6.74,250864,22.99
2,persons-with-significant-control-statement,936034,5.79,367756,39.29
3,individual-beneficial-owner,34980,0.22,3465,9.91
4,legal-person-person-with-significant-control,17395,0.11,6443,37.04
5,corporate-entity-beneficial-owner,15891,0.10,2285,14.38
6,legal-person-beneficial-owner,592,0.00,68,11.49
7,super-secure-person-with-significant-control,386,0.00,100,25.91
8,super-secure-beneficial-owner,169,0.00,7,4.14
9,exemptions,109,0.00,0,0.00


,yil,adet
0,0224,1
1,1961,1
2,1962,2
3,1963,2
4,1964,2
5,1980,1
6,1981,1
7,1987,1
8,1988,2
9,1991,1


In [11]:
uzun = pd.DataFrame(
    [(tur, alan, adet) for (tur, alan), adet in dolu_alan_sayaci.items()],
    columns=["kind", "alan", "adet"],
)
doluluk = (
    uzun.pivot(index="kind", columns="alan", values="adet")
    .reindex(index=[t for t, _ in tur_sayaci.most_common()], columns=ALANLAR)
    .fillna(0)
)
doluluk_yuzde = (doluluk.div(pd.Series(tur_sayaci), axis=0) * 100).round(1)

bilinen = set(ALANLAR) | {"kind", "ceased_on"}
kesif = pd.DataFrame(
    [(tur, anahtar, adet, round(adet / tur_sayaci[tur] * 100, 1))
     for (tur, anahtar), adet in anahtar_sayaci.items()
     if anahtar not in bilinen],
    columns=["kind", "anahtar", "adet", "yuzde"],
).sort_values(["kind", "adet"], ascending=[True, False])

display(doluluk_yuzde)
display(kesif)

alan,name,nationality,country_of_residence,date_of_birth,identification,natures_of_control,notified_on
kind,,,,,,,
corporate-entity-beneficial-owner,100.0,0.0,0.0,0.0,100.0,100.0,100.0
corporate-entity-person-with-significant-control,100.0,0.0,0.0,0.0,100.0,100.0,100.0
exemptions,0.0,0.0,0.0,0.0,0.0,0.0,0.0
individual-beneficial-owner,100.0,100.0,0.0,100.0,0.0,99.9,100.0
individual-person-with-significant-control,100.0,100.0,100.0,100.0,0.0,100.0,100.0
legal-person-beneficial-owner,100.0,0.0,0.0,0.0,100.0,100.0,100.0
legal-person-person-with-significant-control,100.0,0.0,0.0,0.0,99.9,100.0,100.0
persons-with-significant-control-statement,0.0,0.0,0.0,0.0,0.0,0.0,100.0
super-secure-beneficial-owner,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,kind,anahtar,adet,yuzde
20,corporate-entity-beneficial-owner,address,15891,100.0
21,corporate-entity-beneficial-owner,etag,15891,100.0
23,corporate-entity-beneficial-owner,links,15891,100.0
24,corporate-entity-beneficial-owner,principal_office_address,15796,99.4
22,corporate-entity-beneficial-owner,is_sanctioned,15621,98.3
35,corporate-entity-beneficial-owner,identity_verification_details,22,0.1
0,corporate-entity-person-with-significant-control,address,1091153,100.0
1,corporate-entity-person-with-significant-control,etag,1091153,100.0
2,corporate-entity-person-with-significant-control,links,1091153,100.0
36,corporate-entity-person-with-significant-control,identity_verification_details,60,0.0


In [12]:
ifadeler_tablo = pd.DataFrame(ifade_sayaci.most_common(), columns=["ifade", "adet"])
bilinmeyen_tablo = pd.DataFrame(bilinmeyen_ifade.most_common(), columns=["ifade", "adet"])

tur_ozet = pd.DataFrame({
    "kayit": pd.Series(tur_sayaci),
    "ifadesi_olmayan": pd.Series(ifadesi_olmayan),
    "bilinmeyen_ifadeli": pd.Series(bilinmeyenli_kayit),
    "pay_bilgisi_yok": pd.Series(pay_bilgisi_yok),
}).loc[lambda d: d.index.isin(KONTROL_TURLERI)].fillna(0).astype(int)
for sutun in ["ifadesi_olmayan", "bilinmeyen_ifadeli", "pay_bilgisi_yok"]:
    tur_ozet[f"{sutun}_%"] = (tur_ozet[sutun] / tur_ozet["kayit"] * 100).round(2)

print(f"Farklı ifade: {len(ifade_sayaci)} · Tanınmayan farklı ifade: {len(bilinmeyen_ifade)}")
display(tur_ozet)
display(bilinmeyen_tablo)
display(ifadeler_tablo.head(30))

Farklı ifade: 87 · Tanınmayan farklı ifade: 16


,kayit,ifadesi_olmayan,bilinmeyen_ifadeli,pay_bilgisi_yok,ifadesi_olmayan_%,bilinmeyen_ifadeli_%,pay_bilgisi_yok_%
corporate-entity-beneficial-owner,15891,7,314,1829,0.04,1.98,11.51
corporate-entity-person-with-significant-control,1091153,1,6634,58794,0.00,0.61,5.39
individual-beneficial-owner,34980,25,761,7199,0.07,2.18,20.58
individual-person-with-significant-control,14081675,197,9124,1088881,0.00,0.06,7.73
legal-person-beneficial-owner,592,0,7,83,0.00,1.18,14.02
legal-person-person-with-significant-control,17395,0,193,3544,0.00,1.11,20.37


,ifade,adet
0,part-right-to-share-surplus-assets-75-to-100-p...,11091
1,part-right-to-share-surplus-assets-25-to-50-pe...,3543
2,registered-owner-as-nominee-person-england-wal...,835
3,part-right-to-share-surplus-assets-50-to-75-pe...,601
4,part-right-to-share-surplus-assets-75-to-100-p...,486
5,registered-owner-as-nominee-another-entity-eng...,316
6,part-right-to-share-surplus-assets-25-to-50-pe...,164
7,part-right-to-share-surplus-assets-75-to-100-p...,130
8,part-right-to-share-surplus-assets-25-to-50-pe...,94
9,registered-owner-as-nominee-person-scotland-re...,52


,ifade,adet
0,ownership-of-shares-75-to-100-percent,8271307
1,right-to-appoint-and-remove-directors,8111044
2,voting-rights-75-to-100-percent,6578462
3,ownership-of-shares-25-to-50-percent,4708203
4,voting-rights-25-to-50-percent,3808981
5,significant-influence-or-control,1080583
6,ownership-of-shares-50-to-75-percent,568977
7,voting-rights-50-to-75-percent,411233
8,right-to-appoint-and-remove-directors-as-firm,243881
9,significant-influence-or-control-as-firm,238608


In [13]:
huni_tablo = pd.DataFrame(sorted(huni.items()), columns=["adim", "adet"])
huni_tablo["ilk_adima_gore_%"] = (huni_tablo["adet"] / huni_tablo["adet"].iloc[0] * 100).round(2)

basarisiz_tablo = pd.DataFrame(basarisiz_numara.most_common(20), columns=["ham_numara", "adet"])
yabanci_tablo = pd.DataFrame(yabanci_yer.most_common(20), columns=["sicil_yeri", "adet"])

display(huni_tablo)
display(basarisiz_tablo)
display(yabanci_tablo)

,adim,adet,ilk_adima_gore_%
0,1_kurumsal_aktif,853895,100.00
1,2_identification_dolu,853894,100.00
2,3_ingiliz_sicili,747263,87.51
3,4_numara_normalize_edildi,730763,85.58


,ham_numara,adet
0,None,9774
1,N/A,403
2,Not Applicable,123
3,00000000,97
4,England,75
5,.,56
6,355b,44
7,Na,43
8,Tbc,42
9,England And Wales,37


,sicil_yeri,adet
0,<BOŞ>,48407
1,UNITED STATES,5021
2,SWITZERLAND,3813
3,JERSEY,2263
4,JERSEY FINANCIAL SERVICES COMMISSION,2194
5,IRELAND,2144
6,FRANCE,2016
7,GERMANY,1951
8,NETHERLANDS,1758
9,DELAWARE,1425


In [14]:
aciklama_tablo = (
    pd.DataFrame(
        [(aciklama, durum, adet) for (aciklama, durum), adet in aciklama_sayaci.items()],
        columns=["aciklama", "durum", "adet"],
    )
    .pivot(index="aciklama", columns="durum", values="adet")
    .fillna(0).astype(int)
)
aciklama_tablo["toplam"] = aciklama_tablo.sum(axis=1)
aciklama_tablo = aciklama_tablo.sort_values("toplam", ascending=False)

kisi_toplam = sum(uyruk_sayaci.values())
uyruk_tablo = pd.DataFrame(uyruk_sayaci.most_common(30), columns=["uyruk", "adet"])
uyruk_tablo["yuzde"] = (uyruk_tablo["adet"] / kisi_toplam * 100).round(2)

print(f"Farklı uyruk metni: {len(uyruk_sayaci)}")
display(aciklama_tablo)
display(uyruk_tablo)

Farklı uyruk metni: 4683


durum,aktif,sona_ermis,toplam
aciklama,,,
no-individual-or-entity-with-signficant-control,477937,149127,627064
all-beneficial-owners-identified,30239,79120,109359
nobody-has-become-or-ceased-to-be-a-beneficial-owner,29465,53756,83221
steps-to-find-psc-not-yet-completed,5682,23609,29291
psc-details-not-confirmed,7390,18839,26229
psc-exists-but-not-identified,4258,21558,25816
no-beneficial-owner-identified,3237,8501,11738
steps-to-find-psc-not-yet-completed-partnership,2637,2993,5630
somebody-has-become-or-ceased-to-be-a-beneficial-owner,1651,3672,5323


,uyruk,adet,yuzde
0,British,10131260,71.77
1,English,403028,2.85
2,Romanian,255793,1.81
3,Pakistani,248240,1.76
4,Indian,247432,1.75
5,Chinese,229532,1.63
6,Irish,176086,1.25
7,Polish,164347,1.16
8,Italian,133794,0.95
9,Nigerian,115979,0.82


In [15]:
etiket = "tam" if ORNEK_BOYUTU is None else f"ornek_{ORNEK_BOYUTU}"
cikti = Paths().data / "eda" / etiket
cikti.mkdir(parents=True, exist_ok=True)

tablolar = {
    "1_turler": tablo, "2_yillar": yillar, "3_doluluk": doluluk_yuzde, "3_kesif": kesif,
    "4_tur_ozet": tur_ozet, "4_bilinmeyen_ifadeler": bilinmeyen_tablo, "4_ifadeler": ifadeler_tablo,
    "5_huni": huni_tablo, "5_basarisiz_numaralar": basarisiz_tablo, "5_yabanci_yerler": yabanci_tablo,
    "6_aciklamalar": aciklama_tablo, "6_uyruklar": uyruk_tablo,
}
for ad, df in tablolar.items():
    df.to_csv(cikti / f"{ad}.csv", encoding="utf-8")

(cikti / "ozet.json").write_text(json.dumps({
    "dosyalar": [p.name for p in psc_dosyalari],
    "okunan_kayit": toplam,
    "sure_sn": round(time.time() - baslangic, 1),
    "sona_erme_orani_%": round(genel_oran, 2),
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Kaydedildi:", cikti)

Kaydedildi: C:\Users\malpk\Workspace\projects\StajProje\kyb-analyst\data\eda\tam
